In [ ]:
!gdown 1tUjwr64Sgv9DlImHSa5exj0Cv6Ngl6CC

In [ ]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, cross_val_predict, KFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix, mean_squared_error
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv('data_g1.csv')

In [ ]:
df.drop(columns='id', inplace=True)

In [ ]:
df

In [ ]:
df.isnull().sum()

In [ ]:
#dali e balansiran 
df['smoking'].value_counts().plot.bar()

In [ ]:
x = df.drop(columns='smoking')
y = df['smoking']

In [ ]:
#biten cekor - ZA XGBOOST 

from xgboost import XGBClassifier, cv, DMatrix, plot_importance

data_dmatrix = DMatrix(data = x, label = y)

params = { 
    'objective' : "binary:logistic",
    'colsample_bytree' : 0.7,
    'learning_rate' : 0.01,
    'max_depth' : 5
}

cv_results = cv(params=params, 
                dtrain=data_dmatrix, 
                nfold=5, 
                num_boost_round=50, 
                early_stopping_rounds=10, 
                metrics='auc', 
                as_pandas=True, 
                seed=123)

In [ ]:
help(cv)

In [ ]:
help(XGBClassifier)

In [ ]:
cv_results

In [ ]:
xgb_clf = XGBClassifier(**params)

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

In [ ]:
xgb_clf.fit(x_train, y_train)

In [ ]:
plot_importance(xgb_clf)

In [ ]:
from sklearn.model_selection import GridSearchCV

params = {
    'n_estimators' : [100,200,300,700] ,
    'max_depth' : [3,5,7],
    'learning_rate' : [0.1, 0.02, 0.05]
}

grid_search = GridSearchCV(estimator=XGBClassifier(), param_grid=params, cv=10, n_jobs=-1, verbose=0, scoring='roc_auc')

grid_search.fit(x, y)

In [ ]:
estimator = grid_search.best_estimator_

In [ ]:
estimator

In [ ]:
cv = KFold(n_splits=5)

In [ ]:
predictions = cross_val_predict(estimator=estimator,X = x, y=y, cv=cv)

In [ ]:
print(f1_score(y, predictions, average='weighted'))